# 01 · Inventory & registry — one source of truth
Contoso Health (synthetic) runs agents in **Azure (Foundry)**, **AWS (Bedrock, AgentCore)** and **SaaS (Copilot Studio, Snowflake Cortex)**.
`mccp.registry` merges platform adapters into one `AgentRecord` schema and produces the rows that **ServiceNow AI Control Tower** stores as the system of record.

* Local mode: `data/mccp/agents.json` (SIMULATED).
* Live mode: Foundry `AIProjectClient.agents.list()`, Bedrock `list_agents()`, AgentCore `list_agent_runtimes()` (VALIDATE), merged with the fixture/CMDB metadata.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mccp").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

RUN_LIVE = False  # True only after deploying infra/mccp and filling .env (see docs/mccp/PLAN.md)

from mccp.config import Settings
from mccp.report import Console
settings = Settings.from_env(mode="azure" if RUN_LIVE else "local")
c = Console(color=False)
print("mode:", settings.mode, "(SIMULATED)" if settings.is_local else "(LIVE)")

In [ ]:
from mccp.registry import build_inventory
agents, warnings = build_inventory(settings)
c.table(["agent", "platform", "cloud", "owner", "okta app", "gateway"],
        [[a.agent_id, a.platform, a.cloud, a.owner, a.okta_app_id, a.gateway_routed] for a in agents])
print("warnings:", warnings)

## Access reviews — agents managed like teammates
Every agent has an owner, a cost center, an identity (Okta app / Entra Agent ID) and a last access review date.

In [ ]:
from datetime import date
for a in agents:
    age = (date.fromisoformat(settings.demo_date) - date.fromisoformat(a.last_access_review)).days
    print(f"{a.agent_id:<28} owner={a.owner:<36} review {age:>3}d ago {'OVERDUE' if age > 90 else ''}")

## ServiceNow payloads (DRY_RUN unless RUN_LIVE and authorize=True)

In [ ]:
from mccp.receipts import ReceiptWriter
from mccp.servicenow import ServiceNowClient
receipts = ServiceNowClient(settings, ReceiptWriter(settings.receipts_dir / "notebooks")).upsert_agents(agents, authorize=RUN_LIVE)
print([r.status for r in receipts])
c.json(receipts[0].request["body"])